# Which values occur together: beyond the duration curve

A distribution representation, `Distribution()`, fits each attribute's **duration curve**
cluster by cluster: every typical day gets the right *values*. It then still has to decide
*when* each value occurs. tsam offers five orderings for this step.

In this tutorial we reduce two weeks of solar irradiance and electricity load to two typical
days, change only the ordering, and follow one number that a power-system model cares about:
the **residual load**, the load that is left after solar PV.

By the end you will have

* checked that every ordering keeps exactly the same values,
* seen one ordering reproduce the residual load better than another, and
* found three orderings that always pair the same values and differ only in the hours they
  place them at.

This is part 4 of five [duration-curve tutorials](../index.md#duration-curves). Each part stands
on its own.

## 1  Two weeks, two typical days

We take **1–14 April 2010** from the example data: hourly solar irradiance (`GHI`) and
electricity `Load`. Two typical days for two weeks is a deliberately tight budget; it makes
the effect of the ordering easy to see.

Every run in this tutorial uses the same settings, so we write them down once in a small
function. Only its `representation` argument will change. `preserve_column_means=False`
switches off tsam's rescaling step, so that we see the values exactly as the ordering placed
them.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam
from tsam import ClusterConfig, Distribution
from tsam.metrics import series_statistics
from tsam.plot import compare_series, path_panels

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../../data/testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-04-01":"2010-04-14", ["GHI", "Load"]]
UNITS = {"GHI": "W/m²", "Load": "MW", "Residual load": "MW"}
COLORS = {
    "original": "#000000",
    "independent": "#009E73",
    "medoid": "#0072B2",
    "reference": "#AD3977",
    "consensus": "#8759A5",
    "assignment": "#947000",
}


def aggregate(representation):
    return tsam.aggregate(
        data,
        n_clusters=2,
        period_duration="1D",
        cluster=ClusterConfig(method="hierarchical", representation=representation),
        preserve_column_means=False,
    )

Before we reduce anything, look at the two weeks:

In [ ]:
compare_series(
    {"original": data}, units=UNITS, colors=COLORS, title="1 to 14 April 2010"
).show()

Every day the sun rises and sets, on some days more strongly than on others. The load rises
in the morning and stays high into the evening, after the sun has set.

## 2  Same values, different order

tsam offers five orderings. Run all five; `reference` needs an attribute to take its order
from, and we give it the sun (`GHI`). Then compare the larger of the two typical days, the
one that represents 9 of the 14 days:

In [ ]:
orderings = {
    "independent": Distribution(),  # the default
    "medoid": Distribution(concurrency="medoid"),
    "reference": Distribution(concurrency="reference", reference_attribute="GHI"),
    "consensus": Distribution(concurrency="consensus"),
    "assignment": Distribution(concurrency="assignment"),
}
results = {name: aggregate(ordering) for name, ordering in orderings.items()}

counts = results["independent"].cluster_counts
typical = max(counts, key=counts.get)
days = {
    name: result.cluster_representatives.loc[typical]
    for name, result in results.items()
}
compare_series(
    days, units=UNITS, colors=COLORS, title="One typical day, five orderings"
).show()

The five profiles differ: they put the brightest hour and the load peak at different
timesteps.

Now sort each attribute's values from highest to lowest, as a duration curve does:

In [ ]:
compare_series(
    days,
    mode="duration_curve",
    units=UNITS,
    colors=COLORS,
    title="The same typical day, sorted",
).show()

You should see one curve per attribute instead of five: the sorted curves lie exactly on top
of each other. Click the orderings in the legend to hide them one by one; the curve stays
until you hide the last one. We can also check this exactly, for every ordering and both
typical days:

In [ ]:
same_values = all(
    np.array_equal(
        np.sort(result.cluster_representatives.loc[cid], axis=0),
        np.sort(results["independent"].cluster_representatives.loc[cid], axis=0),
    )
    for result in results.values()
    for cid in counts
)
same_values

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert same_values

The ordering moves values around; it never changes them. The duration curves therefore stay
the same whichever ordering we pick. What the ordering does change is **which values occur
at the same time**.

## 3  Measure the residual load

Suppose the region has 400 MW of solar PV: at an irradiance of 1000 W/m² it produces 400 MW.
At every timestep *t*, the **residual load** is the load minus the PV output **at that same
timestep**. It is what the other power plants have to cover:

$$\text{residual load}(t) = \text{Load}(t) - 400\,\text{MW} \times \frac{\text{GHI}(t)}{1000\,\text{W/m}^2}$$

It subtracts one time series from the other, timestep by timestep, so it depends on which
load value and which irradiance value occur at the same timestep. That is exactly what the
ordering decides. Calculate it for the `independent` typical day:

In [ ]:
PV_MW = 400


def residual_load(frame):
    pv = PV_MW * frame["GHI"] / 1000
    return (frame["Load"] - pv).to_frame("Residual load")


day = days["independent"]
balance = pd.DataFrame(
    {
        "Load": day["Load"],
        "PV": PV_MW * day["GHI"] / 1000,
        "Residual load": residual_load(day)["Residual load"],
    }
)
px.line(
    balance,
    labels={"value": "MW", "variable": ""},
    color_discrete_map={"Load": "#999999", "PV": "#E69F00", "Residual load": "#009E73"},
    title="The independent typical day: load minus PV at each timestep",
).show()

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert balance["Residual load"].idxmin() == balance["PV"].idxmax()
assert balance["PV"].loc[balance["Residual load"].idxmax()] == 0

At every timestep, the green line is the gap between the other two: load minus PV. It is
lowest around midday, where the PV produces most, and highest in the evening, when the sun
has set but the load is still high. Another ordering pairs other load values with these PV
values, and the green line changes with it.

Now do the same for the whole two weeks. `result.reconstructed` lays the typical days back
out over the two weeks, one copy for every day they represent. Subtract the PV from the load
at each of the 336 hours, for the original and for `independent` and `medoid`, and sort the
results into duration curves:

In [ ]:
residual = {
    "original": residual_load(data),
    "independent": residual_load(results["independent"].reconstructed),
    "medoid": residual_load(results["medoid"].reconstructed),
}
compare_series(
    residual,
    mode="duration_curve",
    reference="original",
    units=UNITS,
    colors=COLORS,
    title="Residual load over the two weeks",
).show()

Look at the right end of the curves, the hours with the lowest residual load. The original
drops furthest, and `medoid` comes closer to it than `independent`. The table gives the
numbers:

In [ ]:
series_statistics(residual).round(1)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
lowest = {name: frame["Residual load"].min() for name, frame in residual.items()}
assert round(lowest["original"]) == 139
assert round(lowest["medoid"]) == 165
assert round(lowest["independent"]) == 193
means = [frame["Residual load"].mean() for frame in residual.values()]
np.testing.assert_allclose(means, means[0])

In the original two weeks the residual load drops to 139 MW. With `medoid` it reaches
165 MW, with `independent` only 193 MW. Notice that the mean is the same in every row: the
values are the same, only their pairing differs.

## 4  Tie every attribute to the sun

The `reference` ordering takes the order of one attribute and gives it to all the others.
We gave it the sun. Compare its typical day with `independent`:

In [ ]:
compare_series(
    {name: days[name] for name in ["independent", "reference"]},
    units=UNITS,
    colors=COLORS,
    title="The load now follows the sun",
).show()

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert days["reference"]["Load"].idxmax() == days["reference"]["GHI"].idxmax()

Look at the load. Its peak now sits exactly at the brightest hour. At night, where GHI is
zero and tells tsam nothing about the order, the load simply climbs hour by hour. At sunset
it drops sharply instead of staying high into the evening.

What does this do to the residual load?

In [ ]:
residual["reference"] = residual_load(results["reference"].reconstructed)

compare_series(
    residual,
    mode="duration_curve",
    reference="original",
    units=UNITS,
    colors=COLORS,
    title="Residual load over the two weeks",
).show()

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
spread = {
    name: frame["Residual load"].max() - frame["Residual load"].min()
    for name, frame in residual.items()
}
assert min(spread, key=spread.get) == "reference"
ref, orig = (
    residual["reference"]["Residual load"],
    residual["original"]["Residual load"],
)
assert ref.max() < orig.max() and ref.min() > orig.min()

The `reference` curve is the flattest: its residual load never rises as high or drops as low
as the original's. Strong sun now always comes with high load. An hour with strong sun and
low load, which gives the lowest residual load, can no longer occur.

Draw the pairs directly. Each panel shows one typical day as a path through the GHI–Load
plane, hour by hour, in front of the 9 original days it represents (grey):

In [ ]:
members = {
    data.index[day * 24].strftime("%d %b"): data.iloc[
        day * 24 : (day + 1) * 24
    ].reset_index(drop=True)
    for day in np.flatnonzero(results["independent"].cluster_assignments == typical)
}
path_panels(
    {name: days[name] for name in ["independent", "medoid", "reference"]},
    "GHI",
    "Load",
    background=members,
    background_label="original days",
    background_markers=False,
    units=UNITS,
    colors={**COLORS, **dict.fromkeys(members, "#b4b4b4")},
    label_steps=False,
    n_cols=3,
    title="Which load comes with which irradiance?",
).show()

With `reference`, every point lies on one rising curve: more sun always comes with more load,
and the night hours stack up on the load axis. The original days look different. They form
loops, because the load stays high into the evening after the sun has set. `medoid` keeps
such a loop: it borrows the order of one of these real days.

## 5  Three shared orders, the same pairs

`consensus` and `assignment` also give all attributes one shared order, but they choose that
order differently. Add their residual load to the table:

In [ ]:
for name in ["consensus", "assignment"]:
    residual[name] = residual_load(results[name].reconstructed)

series_statistics(residual).round(1)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
def pairs(result):
    return sorted(map(tuple, result.cluster_representatives.round(9).to_numpy()))


assert (
    pairs(results["reference"])
    == pairs(results["consensus"])
    == pairs(results["assignment"])
)

The last three rows are identical. A shared order always places the lowest GHI value in the
same hour as the lowest load value, the second lowest with the second lowest, and so on.
Whichever order is chosen, the pairs are the same, and the residual load depends only on
the pairs.

The hours are not the same, though:

In [ ]:
compare_series(
    {name: days[name] for name in ["reference", "consensus", "assignment"]},
    units=UNITS,
    colors=COLORS,
    title="Same pairs, different hours",
).show()

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert days["consensus"]["GHI"].idxmax() == 11
assert days["reference"]["GHI"].idxmax() == 9

`consensus` places the brightest hour at timestep 11 instead of 9. `consensus` and
`assignment` both keep the load high after sunset and let it fall towards the end of the
day, as in the original days; `reference` cannot, because the sun gives no order at night.
A model that cares about the order of hours, for example one with storage, would see these
three typical days differently.

## 6  Let tsam measure the pairing

The residual load needed an assumption about the PV capacity. tsam also measures how well
the pairing survives without one: `result.concurrency.correlation_error` compares the
correlation between the attributes in the original and in the reconstructed two weeks.
Lower is better.

In [ ]:
pd.Series(
    {name: result.concurrency.correlation_error for name, result in results.items()},
    name="correlation error",
).round(3)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
errors = {
    name: result.concurrency.correlation_error for name, result in results.items()
}
assert errors["medoid"] < errors["independent"] < errors["reference"]
np.testing.assert_allclose(
    [errors["consensus"], errors["assignment"]], errors["reference"]
)

You should see the same ranking as with the residual load: `medoid` lowest, then
`independent`, and the three shared orders highest, all with the same value.

## What you've learned

* The ordering never changes the values. You sorted the typical day under all five
  orderings and found the same values, so every ordering keeps the same duration curves.
* The ordering decides **which values occur together**. On these two weeks, `medoid` came
  closest to the original residual load, and tying the load to the sun made it too flat.
  On other data the ranking can change, so measure it on yours.
* `reference`, `consensus` and `assignment` always form **the same pairs**. They differ only
  in **the hours** they place them at.
* `result.concurrency.correlation_error` measures the pairing without a model of your own.

Where to go next:

* [Choose how attributes coincide](../../how-to/representations.ipynb#choose-how-attributes-coincide)
  — pick an ordering for your own model.
* [Part 1: Fitting a duration curve](01_fitting.ipynb) — how the fitted values are built.
* [Part 2: Local or global](02_scope.ipynb) — fit each typical day's own duration curve, or
  the whole series'.
* [Part 3: Keeping the extremes](03_extremes.ipynb) — keep the real highest and lowest values.
* [Part 5: Making peaks coincide on purpose](05_coincident_peaks.ipynb) — when pairing the
  highest values with each other is exactly what you want.